In [14]:
import torch
from PIL import Image
import pandas as pd
from transformers import AutoProcessor, Blip2Processor, Blip2ForImageTextRetrieval,  Blip2ForConditionalGeneration, BitsAndBytesConfig
from tqdm.auto import tqdm
import os
import numpy as np

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-flan-t5-xl")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [55]:
df = pd.read_csv("metadata.csv")

df = df.dropna(subset=["Image File", "Category"]) # drop rows with missing values
# use the first 100, can be changed for other subsets
df = df[:10]
true_labels = list(df["Category"])
categories = list(df["Category"].unique())

image_paths = [
    os.path.join("images2", fname) # change images2 to name of folder with images
    for fname in df["Image File"]
]

In [56]:
# TODO: add danish to english translation and back
prompt = f"Given an image, output ONLY ONE Category from this list: {', '.join(categories)}. "
prompt

'Given an image, output ONLY ONE Category from this list: Fritid & Have, Lamper, Elektronik, Boligting, Musik & Bøger. '

In [57]:
def classify_image(image_path: str, prompt: str):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, text=prompt, return_tensors="pt", padding=True)
    
    generated_ids = model.generate(  
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    prediction = processor.decode(
        generated_ids[0], skip_special_tokens=True
    )
    print(prediction)
    return prediction

In [58]:
correct = 0

for img_path, gt in zip(image_paths, true_labels):
    pred = classify_image(img_path, prompt)
    if pred == gt:
        correct += 1

accuracy = correct / len(image_paths)
print("Accuracy:", accuracy)

Fritid & Have
Lamper
Elektronik
Fritid & Have
Lamper
Lamper
Boligting
Boligting
Lamper
Elektronik
Accuracy: 0.5
